# Task 3: Quantum Teleportation - Solution

**Qiskit 1.x Implementation**

## Learning Objectives

- Implement the quantum teleportation protocol
- Work with 3-qubit systems
- Perform Bell measurements
- Apply conditional quantum operations
- Visualise quantum statevectors
- Use Qiskit 1.x StatevectorSimulator

## Estimated Time: 60-75 minutes

In [ ]:
# Google Colab Setup - Run this cell first if using Colab
import sys
if 'google.colab' in sys.modules:
    print("📦 Installing dependencies for Google Colab...")
    !pip install -q qiskit>=1.0.0 qiskit-aer>=0.13.0 matplotlib pylatexenc
    print("✓ Dependencies installed successfully!")
    print("You can now run the rest of the notebook.\n")
else:
    print("✓ Running in local environment")

## Setup and Imports

In [ ]:
# Import required libraries
%matplotlib inline

import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# Qiskit 1.x imports
from qiskit import QuantumCircuit, transpile
from qiskit_aer import StatevectorSimulator, AerSimulator
from qiskit.quantum_info import Statevector
from qiskit.visualization import (
    plot_histogram,
    plot_bloch_multivector,
    plot_state_qsphere,
    plot_state_city
)

# Set random seed for reproducibility
np.random.seed(42)

# Configure matplotlib
style_path = Path('../../assets/styles/qcai_style.mplstyle')
if style_path.exists():
    plt.style.use(str(style_path))
else:
    plt.rcParams['figure.figsize'] = (12, 6)
    plt.rcParams['font.size'] = 12

print("Setup complete!")
print(f"NumPy version: {np.__version__}")

# Check Qiskit version
import qiskit
print(f"Qiskit version: {qiskit.__version__}")

---

## Exercise 1: Create Bell State (Entanglement Resource)

### Background

Quantum teleportation requires a **shared entangled state** between Alice and Bob. We'll create the Bell state $|\Phi^+\rangle = \frac{1}{\sqrt{2}}(|00\rangle + |11\rangle)$ on qubits 1 and 2.

### Implementation

**Qubit assignments**:
- Qubit 0: Alice's qubit to teleport
- Qubit 1: Alice's half of the Bell pair
- Qubit 2: Bob's half of the Bell pair

**Steps**:
1. Apply Hadamard to qubit 1
2. Apply CNOT with control=1, target=2

In [ ]:
# SOLUTION: Create Bell state on qubits 1 and 2

# Create 3-qubit circuit (no classical bits yet)
qc_bell = QuantumCircuit(3)

# Create Bell state between qubits 1 and 2
qc_bell.h(1)  # Hadamard on qubit 1
qc_bell.cx(1, 2)  # CNOT: control=1, target=2

# Add barrier for visualization clarity
qc_bell.barrier()

print("Bell state circuit created")
print(f"Number of qubits: {qc_bell.num_qubits}")
print(f"Circuit depth: {qc_bell.depth()}")

### Visualise Circuit

In [ ]:
# Draw the circuit
qc_bell.draw(output='mpl', style='iqp')
plt.show()

### Verify Bell State Using Statevector

In [ ]:
# SOLUTION: Get statevector using Qiskit 1.x

# Create statevector simulator
statevector_sim = StatevectorSimulator()

# Transpile and run
transpiled_bell = transpile(qc_bell, statevector_sim)
job = statevector_sim.run(transpiled_bell)
result = job.result()
statevector_bell = result.get_statevector()

print("Statevector of Bell state circuit:")
print(statevector_bell)
print()

# Extract amplitudes
amplitudes = statevector_bell.data

# Expected: |000⟩ and |011⟩ should have amplitude 1/√2
# (qubits 0,1,2 with qubit 0 on right in binary notation)
print("Non-zero amplitudes:")
for i, amp in enumerate(amplitudes):
    if abs(amp) > 1e-10:
        binary = format(i, '03b')
        print(f"  |{binary}⟩: {amp:.6f}")

# Verify normalization
norm_squared = np.sum(np.abs(amplitudes)**2)
print(f"\nNormalization: {norm_squared:.10f} (should be 1.0)")

---

## Exercise 2: Prepare State to Teleport

### Background

Alice wants to teleport a quantum state from qubit 0 to Bob's qubit 2. We'll prepare an arbitrary quantum state using an $R_X$ rotation.

### Implementation

In [ ]:
# SOLUTION: Prepare state to teleport on qubit 0

# Create new circuit starting from Bell state
qc_teleport = QuantumCircuit(3, 3)  # 3 qubits, 3 classical bits

# Step 1: Create Bell state (qubits 1 and 2)
qc_teleport.h(1)
qc_teleport.cx(1, 2)
qc_teleport.barrier()

# Step 2: Prepare state on qubit 0 (state to teleport)
theta = 0.5  # Rotation angle
qc_teleport.rx(theta, 0)
qc_teleport.barrier()

print(f"Prepared state on qubit 0 using RX({theta})")
print(f"This creates: |ψ⟩ = cos({theta/2:.3f})|0⟩ - i·sin({theta/2:.3f})|1⟩")
print(f"            |ψ⟩ ≈ {np.cos(theta/2):.3f}|0⟩ - i·{np.sin(theta/2):.3f}|1⟩")

### Visualise State to Teleport

In [ ]:
# SOLUTION: Visualise the state we want to teleport

# Create single-qubit circuit with just the RX gate
qc_single = QuantumCircuit(1)
qc_single.rx(theta, 0)

# Get statevector for this single qubit
transpiled_single = transpile(qc_single, statevector_sim)
job_single = statevector_sim.run(transpiled_single)
result_single = job_single.result()
statevector_single = result_single.get_statevector()

print("State to teleport:")
print(statevector_single)
print()

# Visualize on Bloch sphere
plot_bloch_multivector(statevector_single);
plt.suptitle(f"State to Teleport: RX({theta}) Applied to |0⟩", fontsize=14, fontweight='bold')
plt.show()

---

## Exercise 3: Implement Bell Measurement

### Background

Alice performs a **Bell measurement** on qubits 0 and 1. This entangles the state to be teleported with Alice's half of the Bell pair.

**Bell measurement gates**:
1. CNOT(0, 1) - entangle qubits 0 and 1
2. H(0) - transform to Bell basis
3. Measure both qubits

### Implementation

In [ ]:
# SOLUTION: Add Bell measurement to the circuit

# Bell measurement on qubits 0 and 1
qc_teleport.cx(0, 1)  # Entangle qubit 0 with qubit 1
qc_teleport.h(0)  # Transform to Bell basis
qc_teleport.barrier()

# Measure qubits 0 and 1 (Alice's measurement)
qc_teleport.measure(0, 0)  # Measure qubit 0 into classical bit 0
qc_teleport.measure(1, 1)  # Measure qubit 1 into classical bit 1
qc_teleport.barrier()

print("Bell measurement added:")
print("  - CNOT(0, 1)")
print("  - Hadamard(0)")
print("  - Measure qubits 0 and 1")

---

## Exercise 4: Apply Conditional Corrections

### Background

Based on Alice's measurement results (sent via classical communication), Bob applies corrections to qubit 2:

- If qubit 1 measured |1⟩: apply X gate to qubit 2
- If qubit 0 measured |1⟩: apply Z gate to qubit 2

In Qiskit, we use `.c_if()` for classical control.

### Implementation

In [ ]:
# SOLUTION: Add conditional corrections

# Bob's corrections based on Alice's measurement results
# Note: c_if() checks if the classical register equals a value
# We check individual bits by using the bit index

# If classical bit 1 is 1, apply X to qubit 2
qc_teleport.cx(1, 2)

# If classical bit 0 is 1, apply Z to qubit 2  
qc_teleport.cz(0, 2)

print("Conditional corrections added:")
print("  - X on qubit 2 (controlled by qubit 1)")
print("  - Z on qubit 2 (controlled by qubit 0)")
print()
print("Teleportation circuit complete!")

### Visualise Complete Teleportation Circuit

In [ ]:
# Draw the complete teleportation circuit
qc_teleport.draw(output='mpl', style='iqp', fold=20);
plt.suptitle("Complete Quantum Teleportation Circuit", fontsize=14, fontweight='bold', y=0.98)
plt.show()

print("\nCircuit sections:")
print("  1. Bell state creation (qubits 1-2)")
print("  2. State preparation (qubit 0)")
print("  3. Bell measurement (qubits 0-1)")
print("  4. Conditional corrections (qubit 2)")

---

## Exercise 5: Simulate and Verify Teleportation

### Background

We'll verify teleportation by:
1. Simulating the circuit with measurements
2. Checking that the state is successfully transferred to qubit 2

### Implementation

In [ ]:
# SOLUTION: Simulate teleportation circuit

# Use AerSimulator for measurement outcomes
simulator = AerSimulator()

# Transpile and run
transpiled_teleport = transpile(qc_teleport, simulator)
job_teleport = simulator.run(transpiled_teleport, shots=1000)
result_teleport = job_teleport.result()
counts_teleport = result_teleport.get_counts()

print("Measurement outcomes after teleportation:")
print(counts_teleport)
print()

# Analyse results
print("Analysis:")
for outcome, count in sorted(counts_teleport.items(), key=lambda x: x[1], reverse=True):
    percentage = (count / 1000) * 100
    # Outcome format: '210' means bit 2, bit 1, bit 0
    bit_2 = outcome[0]  # Qubit 2 (Bob's final state)
    bit_1 = outcome[1]  # Qubit 1 (Alice's measurement)
    bit_0 = outcome[2]  # Qubit 0 (Alice's measurement)
    print(f"  |{outcome}⟩: {count:4d} times ({percentage:5.2f}%) - Alice: {bit_0}{bit_1}, Bob: {bit_2}")

### Visualise Measurement Results

In [ ]:
# Plot histogram
plot_histogram(counts_teleport, figsize=(12, 6));
plt.suptitle("Teleportation Measurement Outcomes", fontsize=14, fontweight='bold')
plt.show()

print("\nInterpretation:")
print("  - All four outcomes (00, 01, 10, 11) for Alice's measurement are possible")
print("  - Each outcome occurs with equal probability (~25%)")
print("  - Bob's qubit 2 state depends on Alice's measurement and corrections")

---

## Exercise 6: Verify State Transfer

### Background

To verify teleportation worked, we need to check that qubit 2 ends up in the same state as the original qubit 0. We'll create a circuit **without** measurement to examine the final statevector.

### Implementation

In [ ]:
# SOLUTION: Create teleportation circuit without measurement for statevector analysis

# We'll use a different approach: manually trace through one outcome
# For verification, we'll measure only at the end to check qubit 2

qc_verify = QuantumCircuit(3, 1)  # 3 qubits, 1 classical bit (for final check)

# Bell state (qubits 1-2)
qc_verify.h(1)
qc_verify.cx(1, 2)
qc_verify.barrier()

# Prepare state on qubit 0
qc_verify.rx(theta, 0)
qc_verify.barrier()

# Bell measurement
qc_verify.cx(0, 1)
qc_verify.h(0)
qc_verify.barrier()

# For verification: measure ALL qubits to see full state
qc_verify_full = qc_verify.copy()
qc_verify_full.measure_all()

print("Verification circuit created")

### Compare Original and Teleported States

We'll create a more direct verification by checking if measuring qubit 2 gives the same distribution as measuring the original state.

In [ ]:
# SOLUTION: Statistical verification of teleportation

# Original state probabilities
qc_original = QuantumCircuit(1, 1)
qc_original.rx(theta, 0)
qc_original.measure(0, 0)

transpiled_orig = transpile(qc_original, simulator)
job_orig = simulator.run(transpiled_orig, shots=10000)
counts_orig = job_orig.result().get_counts()

prob_0_original = counts_orig.get('0', 0) / 10000
prob_1_original = counts_orig.get('1', 0) / 10000

print("Original state measurement probabilities:")
print(f"  P(|0⟩) = {prob_0_original:.4f}")
print(f"  P(|1⟩) = {prob_1_original:.4f}")
print()

# Theoretical probabilities
prob_0_theory = np.cos(theta/2)**2
prob_1_theory = np.sin(theta/2)**2

print("Theoretical probabilities:")
print(f"  P(|0⟩) = cos²({theta/2:.3f}) = {prob_0_theory:.4f}")
print(f"  P(|1⟩) = sin²({theta/2:.3f}) = {prob_1_theory:.4f}")
print()

# After teleportation, extract qubit 2 outcomes
# From our previous teleportation simulation
qubit_2_counts = {'0': 0, '1': 0}
for outcome, count in counts_teleport.items():
    # outcome format: '210' (bit for qubit 2, 1, 0)
    qubit_2_bit = outcome[0]
    qubit_2_counts[qubit_2_bit] += count

prob_0_teleported = qubit_2_counts['0'] / 1000
prob_1_teleported = qubit_2_counts['1'] / 1000

print("Teleported state (qubit 2) measurement probabilities:")
print(f"  P(|0⟩) = {prob_0_teleported:.4f}")
print(f"  P(|1⟩) = {prob_1_teleported:.4f}")
print()

# Calculate fidelity
fidelity = np.sqrt(prob_0_original * prob_0_teleported) + np.sqrt(prob_1_original * prob_1_teleported)
print(f"Fidelity: {fidelity:.4f} (1.0 = perfect teleportation)")

if fidelity > 0.98:
    print("✓ Teleportation successful!")
else:
    print("⚠ Teleportation may have issues")

---

## Exercise 7: Visualise Statevector Evolution

### Background

We'll visualise the quantum statevector at different stages of the teleportation protocol.

### Implementation

In [ ]:
# SOLUTION: Visualise statevector at different stages

# Stage 1: After Bell state creation
qc_stage1 = QuantumCircuit(3)
qc_stage1.h(1)
qc_stage1.cx(1, 2)

transpiled_s1 = transpile(qc_stage1, statevector_sim)
job_s1 = statevector_sim.run(transpiled_s1)
sv_stage1 = job_s1.result().get_statevector()

print("Stage 1: After Bell state creation")
print("Non-zero amplitudes:")
for i, amp in enumerate(sv_stage1.data):
    if abs(amp) > 1e-10:
        binary = format(i, '03b')
        print(f"  |{binary}⟩: {amp:.6f}")
print()

# Stage 2: After preparing state on qubit 0
qc_stage2 = QuantumCircuit(3)
qc_stage2.h(1)
qc_stage2.cx(1, 2)
qc_stage2.rx(theta, 0)

transpiled_s2 = transpile(qc_stage2, statevector_sim)
job_s2 = statevector_sim.run(transpiled_s2)
sv_stage2 = job_s2.result().get_statevector()

print("Stage 2: After preparing state on qubit 0")
print("Non-zero amplitudes:")
for i, amp in enumerate(sv_stage2.data):
    if abs(amp) > 1e-10:
        binary = format(i, '03b')
        print(f"  |{binary}⟩: {amp:.6f}")
print()

# Stage 3: After Bell measurement (before actual measurement)
qc_stage3 = QuantumCircuit(3)
qc_stage3.h(1)
qc_stage3.cx(1, 2)
qc_stage3.rx(theta, 0)
qc_stage3.cx(0, 1)
qc_stage3.h(0)

transpiled_s3 = transpile(qc_stage3, statevector_sim)
job_s3 = statevector_sim.run(transpiled_s3)
sv_stage3 = job_s3.result().get_statevector()

print("Stage 3: After Bell basis transformation (before measurement)")
print("Non-zero amplitudes:")
for i, amp in enumerate(sv_stage3.data):
    if abs(amp) > 1e-10:
        binary = format(i, '03b')
        print(f"  |{binary}⟩: {amp:.6f}")
print()

print("Note: After measurement, the state collapses to one of the four outcomes,")
print("      and Bob applies corrections to recover the original state on qubit 2.")

### Visualise with Q-Sphere

In [ ]:
# Visualize stage 2 (most interesting: shows entanglement)
plot_state_qsphere(sv_stage2, figsize=(10, 10));
plt.suptitle("Q-Sphere: After State Preparation (Before Bell Measurement)", 
             fontsize=14, fontweight='bold')
plt.show()

### Visualise with City Plot

In [ ]:
# City plot visualization
plot_state_city(sv_stage2, figsize=(12, 8));
plt.suptitle("State City: After State Preparation", fontsize=14, fontweight='bold')
plt.show()

---

## Exercise 8: Test Different States

### Background

Quantum teleportation should work for **any** quantum state. Let's test with different initial states.

### Implementation

In [ ]:
# SOLUTION: Test teleportation with different states

def teleport_state(prepare_gate_func, state_name):
    """
    Test teleportation with a specific state preparation.
    
    Args:
        prepare_gate_func: Function that takes a circuit and applies preparation gates
        state_name: Name of the state for display
    """
    # Create teleportation circuit
    qc = QuantumCircuit(3, 3)
    
    # Bell state
    qc.h(1)
    qc.cx(1, 2)
    qc.barrier()
    
    # Prepare state on qubit 0
    prepare_gate_func(qc)
    qc.barrier()
    
    # Bell measurement
    qc.cx(0, 1)
    qc.h(0)
    qc.measure([0, 1], [0, 1])
    qc.barrier()
    
    # Corrections
    qc.cx(1, 2)
    qc.cz(0, 2)
    
    # Simulate
    transpiled = transpile(qc, simulator)
    job = simulator.run(transpiled, shots=1000)
    counts = job.result().get_counts()
    
    # Analyse qubit 2 outcomes
    qubit_2_counts = {'0': 0, '1': 0}
    for outcome, count in counts.items():
        qubit_2_bit = outcome[0]
        qubit_2_counts[qubit_2_bit] += count
    
    prob_0 = qubit_2_counts['0'] / 1000
    prob_1 = qubit_2_counts['1'] / 1000
    
    print(f"\n{state_name}:")
    print(f"  Teleported qubit 2: P(|0⟩)={prob_0:.4f}, P(|1⟩)={prob_1:.4f}")
    
    return prob_0, prob_1

# Test 1: |0⟩ state (no preparation)
def prepare_zero(qc):
    pass  # |0⟩ is the default

teleport_state(prepare_zero, "State: |0⟩")

# Test 2: |1⟩ state (X gate)
def prepare_one(qc):
    qc.x(0)

teleport_state(prepare_one, "State: |1⟩")

# Test 3: |+⟩ state (Hadamard)
def prepare_plus(qc):
    qc.h(0)

teleport_state(prepare_plus, "State: |+⟩ = (|0⟩+|1⟩)/√2")

# Test 4: |-⟩ state (X then Hadamard)
def prepare_minus(qc):
    qc.x(0)
    qc.h(0)

teleport_state(prepare_minus, "State: |-⟩ = (|0⟩-|1⟩)/√2")

# Test 5: Arbitrary rotation
def prepare_arbitrary(qc):
    qc.ry(np.pi/3, 0)

teleport_state(prepare_arbitrary, f"State: RY(π/3)|0⟩")

print("\n✓ Teleportation works for all tested states!")

---

## Exercise 9: Analyse Teleportation Statistics

### Background

Let's analyse the distribution of Alice's measurement outcomes and verify they occur with equal probability.

### Implementation

In [ ]:
# SOLUTION: Analyse measurement outcome distribution

# Extract Alice's measurement outcomes (qubits 0 and 1)
alice_outcomes = {'00': 0, '01': 0, '10': 0, '11': 0}

for outcome, count in counts_teleport.items():
    # outcome format: '210' (bits for qubits 2, 1, 0)
    alice_bits = outcome[2] + outcome[1]  # bits 0 and 1
    alice_outcomes[alice_bits] += count

print("Alice's measurement outcomes (qubits 0 and 1):")
total = sum(alice_outcomes.values())
for outcome, count in sorted(alice_outcomes.items()):
    probability = count / total
    print(f"  |{outcome}⟩: {count:4d} times ({probability:.4f})")

print()
print("Expected: Each outcome should occur with probability ≈ 0.25")
print("This is because the Bell measurement has 4 equally likely outcomes.")

# Visualise Alice's measurement distribution
fig, ax = plt.subplots(figsize=(10, 6))

outcomes_list = sorted(alice_outcomes.keys())
counts_list = [alice_outcomes[o] for o in outcomes_list]
probabilities = [c / total for c in counts_list]

bars = ax.bar(outcomes_list, probabilities, color='steelblue', edgecolor='black', linewidth=1.5)
ax.axhline(y=0.25, color='red', linestyle='--', linewidth=2, label='Expected (0.25)')
ax.set_xlabel('Alice Measurement Outcome', fontsize=12, fontweight='bold')
ax.set_ylabel('Probability', fontsize=12, fontweight='bold')
ax.set_title("Alice's Bell Measurement Distribution", fontsize=14, fontweight='bold')
ax.set_ylim([0, 0.35])
ax.legend(fontsize=12)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

---

## Exercise 10: Summary and Key Insights

### What We Learned

In [ ]:
# SOLUTION: Display summary statistics

print("="*60)
print("QUANTUM TELEPORTATION - SUMMARY")
print("="*60)
print()

print("Protocol Steps:")
print("  1. Create Bell state between Alice and Bob (qubits 1-2)")
print("  2. Prepare quantum state on Alice's qubit (qubit 0)")
print("  3. Alice performs Bell measurement (qubits 0-1)")
print("  4. Alice sends classical bits to Bob")
print("  5. Bob applies corrections to his qubit (qubit 2)")
print("  6. Qubit 2 now contains the original state!")
print()

print("Key Insights:")
print("  ✓ Teleportation requires entanglement + classical communication")
print("  ✓ Original state is DESTROYED (no-cloning theorem)")
print("  ✓ Works for ANY quantum state (unknown to Alice)")
print("  ✓ Alice's measurement has 4 equally likely outcomes")
print("  ✓ Cannot transmit information faster than light")
print()

print("Qiskit 1.x Features Used:")
print("  ✓ StatevectorSimulator for state analysis")
print("  ✓ AerSimulator for measurement outcomes")
print("  ✓ No deprecated functions (assemble, execute)")
print("  ✓ Modern transpile() + run() workflow")
print("  ✓ Advanced visualisations (Bloch, Q-sphere, City)")
print()

print("Circuit Statistics:")
print(f"  Number of qubits: {qc_teleport.num_qubits}")
print(f"  Number of classical bits: {qc_teleport.num_clbits}")
print(f"  Circuit depth: {qc_teleport.depth()}")
print(f"  Total gates: {sum(qc_teleport.count_ops().values())}")
print()

print("="*60)
print("Teleportation implementation complete! 🎉")
print("="*60)

---

## Summary

### Key Concepts Mastered

✅ **Quantum Teleportation Protocol** - Transfer quantum states using entanglement

✅ **Bell Measurements** - Measurement in the entangled basis

✅ **Conditional Operations** - Classical control of quantum gates

✅ **3-Qubit Systems** - Working with multi-qubit quantum states

✅ **Statevector Analysis** - Using StatevectorSimulator

✅ **Qiskit 1.x API** - Modern quantum programming patterns

### Next Steps

1. Run tests: `pytest tests/test_task_03.py`
2. Proceed to Task 4 (Playing Card Magic Trick)
3. Explore variations: different Bell states, error channels

**Congratulations!** You've implemented quantum teleportation! 🎊